In [5]:
import csv
import re
import os
 
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# NLTK 불용어 데이터 다운로드
nltk.download('stopwords', quiet=True)
 
# -------------------------------------------------------------------
# 1단계: 데이터 로드 및 확인
# -------------------------------------------------------------------
# 파일명 대소문자 주의 (실제 파일명: SMSSpamCollection)
# quoting=csv.QUOTE_NONE: 문자열 안의 " 때문에 여러 줄이 합쳐지는 것을 방지
file_path = './data/raw/text_dataset/SMSSpamCollection'
df = pd.read_csv(
    file_path,
    sep='\t',
    names=['label', 'text_raw'],
    encoding='utf-8',
    quoting=csv.QUOTE_NONE,
)
 
print('=== 1. 데이터 기본 정보 ===')
print(df.info())
print(df.head())
print('행 수:', len(df), '(readme 기준 5,574건)')
print('라벨 분포:\n', df['label'].value_counts())
 
# -------------------------------------------------------------------
# 2단계: 결측치/중복 확인 및 처리, 라벨 인코딩
# -------------------------------------------------------------------
print('\n결측치 개수:\n', df.isnull().sum())
 
n_dup = df.duplicated().sum()
print('중복 행 개수:', n_dup)
df = df.drop_duplicates().reset_index(drop=True)
print('중복 제거 후 행 수:', len(df))
 
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})
 
# -------------------------------------------------------------------
# 3 & 4단계: 텍스트 정제, 불용어 제거, 어근 추출
# -------------------------------------------------------------------
stemmer = PorterStemmer()                           # stemmer 지정(어근 추출 위함)
stop_words = set(stopwords.words('english'))        # 불용어 지정
 
 
def preprocess_text(text):
    text = re.sub(r'\d+', ' number ', text)         # 숫자 -> 토큰 치환(이유:스팸 판별에 강한 단서가 될수있음)
    text = re.sub(r'[^a-z]', ' ', text)             # 알파벳 외 문자(특수문자/구두점) 제거
    words = text.lower().split()                    # 소문자 변환 및 단어 토큰화
    cleaned_words = [                                           
        stemmer.stem(w) for w in words if w not in stop_words   # 불용어 제거 및 어근 추출
    ]
    return ' '.join(cleaned_words)                  # 문자열로 재결합
 
 
df['text_clean'] = df['text_raw'].apply(preprocess_text)
 
n_empty = (df['text_clean'] == '').sum()
print('\n정제 후 빈 문자열 개수:', n_empty)
 
# -------------------------------------------------------------------
# 5단계: 학습/테스트 분할 후 TF-IDF 수치화 (누수 방지)
# -------------------------------------------------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text_clean'],
    df['label_encoded'],
    test_size=0.2,
    stratify=df['label_encoded'],
    random_state=42,
)
 
TFIDF = TfidfVectorizer(max_features=3000)          # 상위 3,000개 단어 추출
X_train = TFIDF.fit_transform(X_train_text)         # 학습 데이터로만 fit
X_test = TFIDF.transform(X_test_text)               # 테스트 데이터는 transform만
 
# -------------------------------------------------------------------
# 결과 확인
# -------------------------------------------------------------------
print('\n=== 전처리 전/후 데이터 비교 ===')
print(df[['label', 'label_encoded', 'text_raw', 'text_clean']].head(3))
print('\nTF-IDF 행렬 크기 (train):', X_train.shape)
print('TF-IDF 행렬 크기 (test):', X_test.shape)


# 저장할 폴더 지정 (없으면 자동 생성)
output_dir = './data/cleaned/cleaned_text_data'
os.makedirs(output_dir, exist_ok=True)

path = os.path.join(output_dir,'SMSSpam_cleaned.csv')
df.to_csv(path, index=False, encoding='utf-8-sig')
 


=== 1. 데이터 기본 정보 ===
<class 'pandas.DataFrame'>
RangeIndex: 5574 entries, 0 to 5573
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   label     5574 non-null   str  
 1   text_raw  5574 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB
None
  label                                           text_raw
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
행 수: 5574 (readme 기준 5,574건)
라벨 분포:
 label
ham     4827
spam     747
Name: count, dtype: int64

결측치 개수:
 label       0
text_raw    0
dtype: int64
중복 행 개수: 403
중복 제거 후 행 수: 5171

정제 후 빈 문자열 개수: 57

=== 전처리 전/후 데이터 비교 ===
  label  label_encoded                                           text_raw  \
0   ham              0  Go until jurong poi